## CrisisLens — Classifier Training
Fine-tunes DistilBERT on HumAID crisis tweets for multi-class classification.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/crisis_classifier', exist_ok=True)
print("Google Drive mounted. Model will save to /content/drive/MyDrive/crisis_classifier")

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import torch, pandas as pd, numpy as np, json, matplotlib.pyplot as plt
from pathlib import Path

c:\Users\ranan\Desktop\CrisisLens — Real-Time Disaster Intelligence Platform\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("data/processed/crisis_dataset.csv")
print(df.shape)
print(df["label"].value_counts())

(6007, 2)
label
medical       2784
flood         1898
fire           667
earthquake     658
Name: count, dtype: int64


In [4]:
labels = sorted(df["label"].unique().tolist())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
Path("data/processed").mkdir(parents=True, exist_ok=True)
with open("data/processed/label_map.json", "w") as f:
    json.dump({"label2id": label2id, "id2label": {str(k): v for k, v in id2label.items()}}, f, indent=2)
df["label_id"] = df["label"].map(label2id)
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label_id"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label_id"], random_state=42)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 4204 | Val: 901 | Test: 902


In [5]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(batch):
    return tokenizer(batch["text"], max_length=128, truncation=True, padding="max_length")
def make_dataset(df):
    ds = Dataset.from_dict({"text": df["text"].tolist(), "label": df["label_id"].tolist()})
    return ds.map(tokenize, batched=True)
train_ds = make_dataset(train_df)
val_ds   = make_dataset(val_df)
test_ds  = make_dataset(test_df)

Map: 100%|██████████| 902/902 [00:00<00:00, 6922.14 examples/s]


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label2id),
    label2id=label2id, id2label=id2label
)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro"),
    }
args = TrainingArguments(
    output_dir="/content/drive/MyDrive/crisis_classifier",
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    fp16=True,
    report_to="none",
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)
print("Starting training on GPU with fp16=True (12-18 min expected)...")
trainer.train()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1709.50it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Users\ranan\Desktop\CrisisLens — Real-Time Disaster Intelligence Platform\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
results = trainer.evaluate(test_ds)
print(results)
preds_output = trainer.predict(test_ds)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids
print(classification_report(y_true, y_pred, target_names=labels))
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig("data/processed/confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
trainer.save_model("/content/drive/MyDrive/crisis_classifier")
tokenizer.save_pretrained("/content/drive/MyDrive/crisis_classifier")
print("✓ Model saved to Google Drive: /MyDrive/crisis_classifier")
print("Download the crisis_classifier folder and place it in models/ in your project")